# Google Scholar author load

Separa la lógica en dos funciones para después copiarla a `nodes.py`:

- `load_google_scholar_parse_author`: toma los HTML raw desde Kedro y arma un DataFrame base.
- `load_google_scholar_author`: normaliza ese DataFrame para cargarlo en `ldg/gs/author`.


In [1]:
import json
from urllib.parse import parse_qs, urlparse
import re

import pandas as pd
from bs4 import BeautifulSoup

pd.set_option('display.max_columns', None)


In [2]:
html_partitions = catalog.load('raw/google_scholar/html')
len(html_partitions), sorted(html_partitions)[:3]


[06/12/26 11:44:08] INFO     Loading data from raw/google_scholar/html (PartitionedDataset)... ]8;id=641076;file:///home/pablo/dev/scholar/kedro-scholar/.venv/lib/python3.10/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=179333;file:///home/pablo/dev/scholar/kedro-scholar/.venv/lib/python3.10/site-packages/kedro/io/data_catalog.py#1048\1048]8;;\

(516, ['Perfiles - UNLP 0', 'Perfiles - UNLP 1', 'Perfiles - UNLP 10'])

In [3]:
def load_google_scholar_author(html_partitions: dict) -> pd.DataFrame:

    def parse_user_id(profile_url: str | None) -> str | None:
        if not profile_url:
            return None
        query = parse_qs(urlparse(profile_url).query)
        users = query.get('user')
        return users[0] if users else None


    def parse_cited_by(raw_text: str | None) -> int | None:
        if not raw_text:
            return None
        match = re.search(r'(\d[\d.,]*)', raw_text)
        if not match:
            return None
        digits = re.sub(r'[^\d]', '', match.group(1))
        return int(digits) if digits else None


    def extract_author_cards(source_file: str, html_text: str) -> list[dict]:
        soup = BeautifulSoup(html_text, 'html.parser')
        rows = []

        for card in soup.select('div.gsc_1usr'):
            name_node = card.select_one('h3.gs_ai_name a')
            affiliation_node = card.select_one('div.gs_ai_aff')
            email_node = card.select_one('div.gs_ai_eml')
            cited_by_node = card.select_one('div.gs_ai_cby')
            interest_nodes = card.select('a.gs_ai_one_int')

            profile_url = name_node.get('href') if name_node else None

            rows.append({
                'source_file': source_file,
                'source_system': 'google_scholar',
                'entity_type': 'author',
                'name': name_node.get_text(' ', strip=True) if name_node else None,
                'profile_url': profile_url,
                'user': parse_user_id(profile_url),
                'affiliation': affiliation_node.get_text(' ', strip=True) if affiliation_node else None,
                'verified_email': email_node.get_text(' ', strip=True) if email_node else None,
                'cited_by': parse_cited_by(cited_by_node.get_text(' ', strip=True) if cited_by_node else None),
                'interests': [node.get_text(' ', strip=True) for node in interest_nodes],
            })

        return rows

    def parse_google_scholar_author(html_partitions: dict) -> pd.DataFrame:
        rows = []
        for partition_id, partition_value in sorted(html_partitions.items()):
            html_text = partition_value() if callable(partition_value) else partition_value
            rows.extend(extract_author_cards(f"{partition_id}.html", html_text))

        df_author_raw = pd.DataFrame(rows).convert_dtypes()
        df_author = (
            df_author_raw.sort_values(["user", "source_file"], na_position="last")
            .drop_duplicates(subset=["user"], keep="first")
            .reset_index(drop=True)
            .copy()
        )

        df_author["interests"] = df_author["interests"].apply(
            lambda values: json.dumps(list(values), ensure_ascii=False)
            if hasattr(values, "__iter__") and not isinstance(values, str)
            else values
        )

        return df_author.convert_dtypes()

    df_author = parse_google_scholar_author(html_partitions)
    df_author["_load_datetime"] = pd.Timestamp.now().floor("s")

    return df_author.convert_dtypes()


In [4]:
df_author = load_google_scholar_author(html_partitions)

summary_raw = {
    'html_files': len(html_partitions),
    'rows_raw': len(df_author),
    'unique_users_raw': df_author['user'].nunique(dropna=True),
}

summary_raw


{'html_files': 516, 'rows_raw': 5150, 'unique_users_raw': 5150}

In [5]:
df_author.head()


,source_file,source_system,entity_type,name,profile_url,user,affiliation,verified_email,cited_by,interests,_load_datetime
0,Perfiles - UNLP 1780.html,google_scholar,author,Vicente Dressino,https://scholar.google.com/citations?hl=es&use...,--g4oSEAAAAJ,Profesor de la Facultad de Ciencias Naturales ...,Dirección de correo verificada de fcnym.unlp.e...,328,"[""theoretical biology"", ""evolution"", ""function...",2026-06-12 11:44:26
1,Perfiles - UNLP 1620.html,google_scholar,author,Verónica Cruz,https://scholar.google.com/citations?hl=es&use...,-1-HplkAAAAJ,Profesora Facultad de Trabajo Social Universid...,Dirección de correo verificada de trabajosocia...,419,"[""trabajo social"", ""ciencias sociales derechos...",2026-06-12 11:44:26
2,Perfiles - UNLP 1630.html,google_scholar,author,Paula Viviana Soza Rossi,https://scholar.google.com/citations?hl=es&use...,-1q84UkAAAAJ,"Licenciada en Sociología, Universidad Nacional...",Dirección de correo verificada de fahce.unlp.e...,416,"[""Estudios de Género"", ""Teoría Social"", ""Epist...",2026-06-12 11:44:26
3,Perfiles - UNLP 5130.html,google_scholar,author,Celeste Lucca,https://scholar.google.com/citations?hl=es&use...,-2r4SjIAAAAJ,Profesora del Taller de Comprensión y Producci...,Dirección de correo verificada de perio.unlp.e...,<NA>,"[""Escritura"", ""Lectura"", ""Edición"", ""Ficción"",...",2026-06-12 11:44:26
4,Perfiles - UNLP 1460.html,google_scholar,author,María Fernanda Achinelly,https://scholar.google.com/citations?hl=es&use...,-3dZqLEAAAAJ,"Investigador Independiente CEPAVE, CONICET-UNLP",Dirección de correo verificada de cepave.unlp....,528,"[""Nematología""]",2026-06-12 11:44:26
